In [ ]:
import os
import json
import random
from google import genai
from google.genai import types

INPUT_FOLDER = r"D:\PJ\spatial_dataset"
OUTPUT_FOLDER = r"D:\PJ\gemini_results"
MODEL_NAME = "gemini-3.1-flash-lite"
API_KEY

if not API_KEY: raise RuntimeError("Please set the GEMINI_API_KEY environment variable before running this script.")
client = genai.Client(api_key=API_KEY)

DIRECTIONS = ["front", "back", "left", "right"]
FOR_NAMES = ["speaker", "intrinsic", "hearer"]


def ask_gemini(question, image_path):
    prompt = (
        question
        + "\nCarefully analyze the image and reason about the spatial relationship before answering."
        + "\nAnswer in one short sentence."
        + "\nAnswer only the question that was asked."
    )
    with open(image_path, "rb") as f: image_bytes = f.read()
    while True:
        try:
            response = client.models.generate_content(model=MODEL_NAME, contents=[types.Part.from_bytes(data=image_bytes, mime_type="image/png"), prompt], config=types.GenerateContentConfig(temperature=0))
            return response.text.strip() if response.text else ""
        except Exception as e:
            if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                print("\nRate limit reached.")
                raise
            else:
                raise


def get_answers(obj): return {"speaker": obj["relations"]["speaker"], "intrinsic": obj["relations"]["intrinsic"], "hearer": obj["relations"]["hearer"]}


def find_slot_by_speaker_direction(objects, direction):
    for role in DIRECTIONS:
        if objects[role]["relations"]["speaker"] == direction: return role
    return None


def classify_relation(answer, valid_relations):
    answer_lower = answer.lower().strip()
    matches = [relation for relation in DIRECTIONS if relation in answer_lower and relation in valid_relations]
    if len(matches) == 1: return matches[0]
    if len(matches) > 1: return matches
    return None


def infer_for_from_relation(answer, answers):
    relation = classify_relation(answer, answers.values())
    if relation is None: return None
    if isinstance(relation, list):
        matching = [for_name for for_name in FOR_NAMES if answers[for_name] in relation]
        return matching if len(matching) > 1 else matching[0] if matching else None
    matching = [for_name for for_name in FOR_NAMES if answers[for_name] == relation]
    if len(matching) == 1: return matching[0]
    if len(matching) > 1: return matching
    return None


def find_color_in_answer(answer, colors):
    answer_lower = answer.lower()
    matches = [c for c in colors if c.lower() in answer_lower]
    if len(matches) == 1: return matches[0]
    if len(matches) > 1: return matches
    return None


def infer_for_from_color(answer, object_type, target_direction, objects):
    candidate_colors = [objects[role]["color"] for role in DIRECTIONS]
    answer_color = find_color_in_answer(answer, candidate_colors)
    if not isinstance(answer_color, str): return {"color": answer_color, "FoR": None, "role": None}
    matching_roles = [role for role in DIRECTIONS if objects[role]["type"] == object_type and objects[role]["color"].lower() == answer_color.lower()]
    if len(matching_roles) != 1: return {"color": answer_color, "FoR": None, "role": matching_roles}
    role = matching_roles[0]
    answers = get_answers(objects[role])
    matching_fors = [for_name for for_name in FOR_NAMES if answers[for_name] == target_direction]
    return {"color": answer_color, "FoR": matching_fors if matching_fors else None, "role": role}


def generate_experiment1(data):
    objects = data["objects"]
    questions = []
    for role in DIRECTIONS:
        obj = objects[role]
        questions.append({"question": f"Where is the {obj['color']} {obj['type']}?", "target": obj["type"], "color": obj["color"], "role": role, "answers": get_answers(obj)})
    return {"experiment": 1, "questions": questions}


def generate_experiment2(data):
    landmark_type = data["landmark"]["type"]
    objects = data["objects"]
    target_role = random.choice(DIRECTIONS)
    obj = objects[target_role]
    object_type, color, answers = obj["type"], obj["color"], get_answers(obj)
    questions = [
        {"question": f"From the speaker's perspective, where is the {color} {object_type} relative to the {landmark_type}?", "target": object_type, "color": color, "FoR": "speaker", "answers": answers},
        {"question": f"From the {landmark_type}'s own orientation, where is the {color} {object_type} relative to the {landmark_type}?", "target": object_type, "color": color, "FoR": "intrinsic", "answers": answers},
        {"question": f"The hearer is standing opposite the speaker. From the hearer's perspective, where is the {color} {object_type} relative to the {landmark_type}?", "target": object_type, "color": color, "FoR": "hearer", "answers": answers}
    ]
    return {"experiment": 2, "landmark": {"type": landmark_type}, "target": {"role": target_role, "type": object_type, "color": color}, "questions": questions}


def generate_experiment3(data):
    landmark_type = data["landmark"]["type"]
    objects = data["objects"]
    right_role = find_slot_by_speaker_direction(objects, "right")
    left_role = find_slot_by_speaker_direction(objects, "left")
    priming = objects["priming"]
    right_item = objects[right_role]
    left_item = objects[left_role]
    evaluation = objects["evaluation"]
    return {
        "experiment": 3,
        "T1": {"question": f"Can you see the {priming['type']} in the scene?", "object": priming["type"], "color": priming["color"], "role": "priming", "answers": get_answers(priming)},
        "T2": {"question": f"What is the color of the {right_item['type']} to the right of the {landmark_type}?", "object": right_item["type"], "color": right_item["color"], "role": right_role, "direction": "right", "answers": get_answers(right_item)},
        "T3": {"question": f"What is the color of the {left_item['type']} to the left of the {landmark_type}?", "object": left_item["type"], "color": left_item["color"], "role": left_role, "direction": "left", "answers": get_answers(left_item)},
        "T4": {"question": f"Where is the {evaluation['type']} relative to the {landmark_type}?", "object": evaluation["type"], "color": evaluation["color"], "role": "evaluation", "answers": get_answers(evaluation)}
    }


def run_experiment1(data, image_path):
    experiment = generate_experiment1(data)
    results = []
    for q in experiment["questions"]:
        answer = ask_gemini(q["question"], image_path)
        detected_for = infer_for_from_relation(answer, q["answers"])
        results.append({"question": q["question"], "target": q["target"], "color": q["color"], "role": q["role"], "gemini_answer": answer, "detected_FoR": detected_for, "expected_answers": q["answers"]})
    return {"experiment": 1, "results": results}


def run_experiment2(data, image_path):
    experiment = generate_experiment2(data)
    results = []
    for q in experiment["questions"]:
        answer = ask_gemini(q["question"], image_path)
        detected_relation = classify_relation(answer, q["answers"].values())
        expected_relation = q["answers"][q["FoR"]]
        correct = detected_relation == expected_relation
        results.append({"question": q["question"], "requested_FoR": q["FoR"], "target": q["target"], "color": q["color"], "gemini_answer": answer, "detected_relation": detected_relation, "expected_relation": expected_relation, "correct": correct})
    return {"experiment": 2, "target": experiment["target"], "results": results}


def run_experiment3(data, image_path):
    experiment = generate_experiment3(data)
    objects = data["objects"]
    t1, t2, t3, t4 = experiment["T1"], experiment["T2"], experiment["T3"], experiment["T4"]

    answer1 = ask_gemini(t1["question"], image_path)

    prompt2 = t1["question"] + "\n" + answer1 + "\n\n" + t2["question"]
    answer2 = ask_gemini(prompt2, image_path)
    t2_info = infer_for_from_color(answer2, t2["object"], t2["direction"], objects)

    prompt3 = prompt2 + "\n" + answer2 + "\n\n" + t3["question"]
    answer3 = ask_gemini(prompt3, image_path)
    t3_info = infer_for_from_color(answer3, t3["object"], t3["direction"], objects)

    prompt4 = prompt3 + "\n" + answer3 + "\n\n" + t4["question"]
    answer4 = ask_gemini(prompt4, image_path)
    t4_for = infer_for_from_relation(answer4, t4["answers"])

    return {
        "experiment": 3,
        "T1": {"question": t1["question"], "gemini_answer": answer1},
        "T2": {"question": t2["question"], "gemini_answer": answer2, "answer_color": t2_info["color"], "answer_role": t2_info["role"], "detected_FoR": t2_info["FoR"], "expected_answers": t2["answers"]},
        "T3": {"question": t3["question"], "gemini_answer": answer3, "answer_color": t3_info["color"], "answer_role": t3_info["role"], "detected_FoR": t3_info["FoR"], "expected_answers": t3["answers"]},
        "T4": {"question": t4["question"], "gemini_answer": answer4, "detected_FoR": t4_for, "expected_answers": t4["answers"]},
        "FoR_sequence": {"T2": t2_info["FoR"], "T3": t3_info["FoR"], "T4": t4_for}
    }


def _bump(counts, detected, ambiguous_key="ambiguous"):
    if isinstance(detected, list):
        if len(detected) == 0: counts["unknown"] += 1
        elif len(detected) > 1:
            if ambiguous_key is not None: counts[ambiguous_key] += 1
        elif detected[0] in FOR_NAMES: counts[detected[0]] += 1
    elif detected in FOR_NAMES: counts[detected] += 1
    else: counts["unknown"] += 1


def calculate_statistics(all_results):
    exp1_counts = {"speaker": 0, "intrinsic": 0, "hearer": 0, "ambiguous": 0, "unknown": 0}
    exp2_stats = {name: {"correct": 0, "total": 0} for name in FOR_NAMES}
    exp3_t2 = {"speaker": 0, "intrinsic": 0, "hearer": 0, "unknown": 0}
    exp3_t3 = {"speaker": 0, "intrinsic": 0, "hearer": 0, "unknown": 0}
    exp3_t4 = {"speaker": 0, "intrinsic": 0, "hearer": 0, "ambiguous": 0, "unknown": 0}
    persistence = {}

    for scene in all_results:
        for result in scene["experiment1"]["results"]: _bump(exp1_counts, result["detected_FoR"])
        for result in scene["experiment2"]["results"]:
            for_name = result["requested_FoR"]
            exp2_stats[for_name]["total"] += 1
            if result["correct"]: exp2_stats[for_name]["correct"] += 1

        sequence = scene["experiment3"]["FoR_sequence"]
        _bump(exp3_t2, sequence["T2"], ambiguous_key=None)
        _bump(exp3_t3, sequence["T3"], ambiguous_key=None)
        _bump(exp3_t4, sequence["T4"])
        key = f"{sequence['T2']}|{sequence['T3']}|{sequence['T4']}"
        persistence[key] = persistence.get(key, 0) + 1

    for name in FOR_NAMES:
        total = exp2_stats[name]["total"]
        correct = exp2_stats[name]["correct"]
        exp2_stats[name]["accuracy"] = correct / total if total else 0

    return {"experiment1": {"FoR_counts": exp1_counts}, "experiment2": {"FoR_accuracy": exp2_stats}, "experiment3": {"T2_FoR_counts": exp3_t2, "T3_FoR_counts": exp3_t3, "T4_FoR_counts": exp3_t4, "FoR_sequences": persistence}}


def get_scene_files():
    scene_files = [f for f in os.listdir(INPUT_FOLDER) if f.startswith("scene_") and f.endswith(".json") and "_experiment" not in f and f not in ("questions.json", "gemini_results.json")]
    scene_files.sort()
    return scene_files


def load_previous_results():
    path = os.path.join(OUTPUT_FOLDER, "gemini_results.json")
    if not os.path.exists(path): return {"dataset": "gemini_spatial_experiment", "model": MODEL_NAME, "num_scenes": 0, "scenes": [], "statistics": {}}
    try:
        with open(path, "r", encoding="utf-8") as f: return json.load(f)
    except Exception:
        return {"dataset": "gemini_spatial_experiment", "model": MODEL_NAME, "num_scenes": 0, "scenes": [], "statistics": {}}


def save_results(all_results):
    statistics = calculate_statistics(all_results)
    output = {"dataset": "gemini_spatial_experiment", "model": MODEL_NAME, "num_scenes": len(all_results), "scenes": all_results, "statistics": statistics}
    path = os.path.join(OUTPUT_FOLDER, "gemini_results.json")
    with open(path, "w", encoding="utf-8") as f: json.dump(output, f, indent=4, ensure_ascii=False)
    return path, statistics


def generate_questions_dataset():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    scene_files = get_scene_files()
    previous_data = load_previous_results()
    all_results = previous_data.get("scenes", [])
    completed_scene_ids = {scene["scene_id"] for scene in all_results}

    print(f"Total scene files: {len(scene_files)}")
    print(f"Already completed: {len(completed_scene_ids)}")

    for index, filename in enumerate(scene_files, 1):
        scene_id = filename.replace(".json", "")
        if scene_id in completed_scene_ids: continue

        json_path = os.path.join(INPUT_FOLDER, filename)
        image_path = os.path.join(INPUT_FOLDER, filename.replace(".json", "_speaker.png"))

        if not os.path.exists(image_path):
            print(f"\nImage not found: {image_path}")
            continue

        with open(json_path, "r", encoding="utf-8") as f: data = json.load(f)

        print(f"\rProcessing {index}/{len(scene_files)} | completed {len(all_results)}/{len(scene_files)} | {scene_id}", end="", flush=True)

        try:
            scene_result = {"scene_id": data["scene_id"], "image": os.path.basename(image_path), "experiment1": run_experiment1(data, image_path), "experiment2": run_experiment2(data, image_path), "experiment3": run_experiment3(data, image_path)}
            all_results.append(scene_result)
            completed_scene_ids.add(scene_id)
            path, _ = save_results(all_results)
            print(f"\rCompleted {len(all_results)}/{len(scene_files)} | {scene_id}", end="", flush=True)
        except Exception as e:
            print(f"\nERROR {scene_id}: {repr(e)}")
            continue

    path, statistics = save_results(all_results)

    print("\n\n" + "=" * 40)
    print("Gemini experiments completed")
    print("=" * 40)
    print("Scenes:", len(all_results))
    print("Output:", path)
    print()
    print(json.dumps(statistics, indent=4, ensure_ascii=False))
    
if __name__ == "__main__":
    generate_questions_dataset()